# Scanpath explorer — one image, one human, one AI sample

Look at a single MIT1003 stimulus and compare:

- DG3's **priority map** for the start fixation
- One human subject's actual scanpath
- An AI scanpath sampled from DG3

Edit the parameters in the cell marked **PARAMETERS** below and re-run.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import deepgaze_pytorch
import pysaliency

from tez_deepgaze.centerbias import load_centerbias_for_image
from tez_deepgaze.device import pick_device, to_device
from tez_deepgaze.human_scanpaths import pick_human_scanpath
from tez_deepgaze.instrument import compute_log_density, sample_scanpath

## PARAMETERS — edit these

In [ ]:
STIM_IDX     = 91     # MIT1003 stimulus index (0..1002)
SUBJECT_IDX  = 0      # which human subject's scanpath to show (0..14 for most stims)
N_FIX        = 8      # how many fixations to sample from the AI
SEED         = 0      # RNG seed for the AI sampler

In [ ]:
device = pick_device()
print(f'device: {device}')

stimuli, fixations = pysaliency.get_mit1003(
    location=str(REPO / 'data' / 'mit1003')
)
model = to_device(deepgaze_pytorch.DeepGazeIII(pretrained=True), device).eval()
print(f'loaded {len(stimuli)} stimuli; DG3 ready')

In [ ]:
image = np.asarray(stimuli.stimuli[STIM_IDX])
if image.ndim == 2:
    image = np.stack([image] * 3, axis=-1)
H, W = image.shape[:2]
cb = load_centerbias_for_image(H, W)

human_xs, human_ys, subj_id = pick_human_scanpath(
    fixations, STIM_IDX, subject_idx=SUBJECT_IDX
)
start_xy = (float(human_xs[0]), float(human_ys[0]))
print(f'image {STIM_IDX}: {W}×{H} px; subject {subj_id}; '
      f'human path has {len(human_xs)} fixations; '
      f'start point ({start_xy[0]:.0f}, {start_xy[1]:.0f})')

In [ ]:
sp = sample_scanpath(model, image, cb, start_xy, N_FIX, device, seed=SEED)
ai_label = 'AI scanpath (vendor DG3)'
print(ai_label)

In [ ]:
# Priority map conditioned on the start fixation only
log_d = compute_log_density(model, image, cb, [start_xy[0]], [start_xy[1]], device)
prio = np.exp(log_d)
prio_disp = prio / (prio.max() + 1e-12)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
for ax in axes:
    ax.imshow(image)
    ax.axis('off')

axes[0].imshow(prio_disp, cmap='inferno', alpha=0.55)
axes[0].set_title('A. DG3 priority map (start fixation)')

axes[1].plot(human_xs, human_ys, '-o', color='#1d6fb8',
             linewidth=1.6, markersize=8, markeredgecolor='white')
axes[1].plot(human_xs[0], human_ys[0], 'o',
             color='#39ff14', markersize=12, markeredgecolor='black')
axes[1].set_title(f'B. Human scanpath (subject {subj_id})')

axes[2].plot(sp.x, sp.y, '-o', color='#e63946',
             linewidth=1.6, markersize=8, markeredgecolor='white')
axes[2].plot(sp.x[0], sp.y[0], 'o',
             color='#39ff14', markersize=12, markeredgecolor='black')
axes[2].set_title(f'C. {ai_label}')

fig.suptitle(f'MIT1003 stim {STIM_IDX} — green dot = start fixation', fontsize=13)
fig.tight_layout()
plt.show()

## Try this

- Change `STIM_IDX` to a different image (0..1002). Easy/hard examples to start from:
  77 (sticker on dashboard — easy), 522 (underwater texture — hard), 91 (airplane).
- Change `SEED` to draw a different AI scanpath — the model output is a
  *distribution*, so any single sampled path is one valid draw.